In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

In [2]:
print("Loading filtered data...")
df = pd.read_csv('filtered_eur_rates.csv')

Loading filtered data...


In [3]:
#Converting date column
df['date'] = pd.to_datetime(df['date'])

In [4]:
#Assuming a standard transfer amount of €1,000
TRANSFER_AMOUNT = 1000

In [5]:
# Defining platform fees and markups (from Phase 1)
PLATFORM_CONFIG = {
    'paypal': {
        'markup_pct': 0.03,      # 3% above base rate
        'fixed_fee_eur': 1.99,   # €1.99 for EEA transfers
    },
    'wise': {
        'markup_pct': 0.00,      # 0% - uses mid-market rate
        'fee_pct': 0.005,        # ~0.5% transfer fee
    },
    'revolut': {
        'markup_pct_weekday': 0.00,   # 0% on weekdays
        'markup_pct_weekend': 0.01,   # 1% on weekends
        'fee_pct': 0.00,              # Free up to €1,000/month
    }
}


In [6]:
# Function to determine day of week (0=Monday, 6=Sunday)
def is_weekend(date):
    return date.dayofweek >= 5  # Saturday=5, Sunday=6

In [7]:
# Calculating for each platform
print("Calculating final costs for all platforms...")

#PayPal
df['paypal_rate'] = df['close'] * (1 + PLATFORM_CONFIG['paypal']['markup_pct'])
df['paypal_final_cost'] = (TRANSFER_AMOUNT * df['paypal_rate']) + (PLATFORM_CONFIG['paypal']['fixed_fee_eur'] * df['close'])
df['paypal_markup'] = df['paypal_rate'] - df['close']

#Wise
df['wise_rate'] = df['close'] * (1 + PLATFORM_CONFIG['wise']['markup_pct'])
df['wise_final_cost'] = (TRANSFER_AMOUNT * df['wise_rate']) + (TRANSFER_AMOUNT * df['wise_rate'] * PLATFORM_CONFIG['wise']['fee_pct'])
df['wise_markup'] = df['wise_rate'] - df['close']

#Revolut
df['revolut_markup_pct'] = df['date'].apply(lambda d: PLATFORM_CONFIG['revolut']['markup_pct_weekend'] if is_weekend(d) else PLATFORM_CONFIG['revolut']['markup_pct_weekday'])
df['revolut_rate'] = df['close'] * (1 + df['revolut_markup_pct'])
df['revolut_final_cost'] = TRANSFER_AMOUNT * df['revolut_rate']
df['revolut_markup'] = df['revolut_rate'] - df['close']

print(f"Calculated costs for {len(df)} rows")

Calculating final costs for all platforms...
Calculated costs for 3845 rows


In [8]:
#Adding Time Variables
print("Adding time variables...")

df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day_of_week'] = df['date'].dt.dayofweek  # 0=Monday, 6=Sunday
df['day_name'] = df['date'].dt.day_name()
df['quarter'] = df['date'].dt.quarter

Adding time variables...


In [9]:
# Season mapping
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Autumn'

df['season'] = df['month'].apply(get_season)

In [10]:
#Cleaning Data
print("Cleaning data...")
initial_rows = len(df)

#Removing duplicates
df = df.drop_duplicates(subset=['slug', 'date'])

#Removing rows with missing critical values
df = df.dropna(subset=['close', 'paypal_final_cost', 'wise_final_cost', 'revolut_final_cost'])

#Removing any rows with negative or zero exchange rates
df = df[df['close'] > 0]

final_rows = len(df)
print(f"Removed {initial_rows - final_rows} rows (duplicates or invalid data)")
print(f"Final cleaned rows: {final_rows}")

Cleaning data...
Removed 0 rows (duplicates or invalid data)
Final cleaned rows: 3845


In [11]:
#Exporting Cleaned Data
print("Exporting cleaned data...")

#Saving as Excel
excel_path = 'cleaned_data.xlsx'
df.to_excel(excel_path, index=False)

#Saving as CSV
csv_path = 'cleaned_data_for_python.csv'
df.to_csv(csv_path, index=False)

Exporting cleaned data...


In [14]:
#Descriptive Statistics
#Selecting numeric columns for summary
numeric_cols = ['close', 'paypal_rate', 'wise_rate', 'revolut_rate',
                'paypal_final_cost', 'wise_final_cost', 'revolut_final_cost',
                'paypal_markup', 'wise_markup', 'revolut_markup']

stats = df[numeric_cols].describe()

In [15]:
# Adding additional stats (total cost difference)
stats.loc['mean_diff_paypal_vs_wise'] = (df['paypal_final_cost'] - df['wise_final_cost']).mean()
stats.loc['mean_diff_paypal_vs_revolut'] = (df['paypal_final_cost'] - df['revolut_final_cost']).mean()
stats.loc['mean_diff_wise_vs_revolut'] = (df['wise_final_cost'] - df['revolut_final_cost']).mean()

print(stats)

                                   close  paypal_rate    wise_rate  \
count                        3845.000000  3845.000000  3845.000000   
mean                           42.960410    44.249223    42.960410   
std                            59.482932    61.267419    59.482932   
min                             0.694100     0.714923     0.694100   
25%                             0.884770     0.911313     0.884770   
50%                             1.124202     1.157928     1.124202   
75%                           121.968002   125.627042   121.968002   
max                           144.929993   149.277893   144.929993   
mean_diff_paypal_vs_wise     1159.501472  1159.501472  1159.501472   
mean_diff_paypal_vs_revolut  1374.303523  1374.303523  1374.303523   
mean_diff_wise_vs_revolut     214.802051   214.802051   214.802051   

                             revolut_rate  paypal_final_cost  wise_final_cost  \
count                         3845.000000        3845.000000      3845.000000 

In [16]:
# Saving to text file
output_path = 'descriptive_stats_output.txt'
with open(output_path, 'w') as f:
    f.write("\n")
    f.write("DESCRIPTIVE STATISTICS - CROSS-BORDER DIGITAL PAYMENTS\n")
    f.write("\n\n")
    f.write(f"Total observations: {len(df)}\n")
    f.write(f"Date range: {df['date'].min()} to {df['date'].max()}\n")
    f.write(f"Currency pairs: {df['slug'].unique().tolist()}\n\n")
    f.write("Transfer amount assumed: €1,000\n\n")
    f.write(stats.to_string())
    f.write("\n\n""\n")
    f.write("KEY FINDINGS:\n")
    f.write("\n")
    avg_paypal = df['paypal_final_cost'].mean()
    avg_wise = df['wise_final_cost'].mean()
    avg_revolut = df['revolut_final_cost'].mean()
    f.write(f"Average PayPal final cost: €{avg_paypal:.2f}\n")
    f.write(f"Average Wise final cost: €{avg_wise:.2f}\n")
    f.write(f"Average Revolut final cost: €{avg_revolut:.2f}\n")
    f.write(f"Average savings (Wise vs PayPal): €{avg_paypal - avg_wise:.2f}\n")
    f.write(f"Average savings (Revolut vs PayPal): €{avg_paypal - avg_revolut:.2f}\n")